# 06 · Nemori × A-MEM — 합성 동작 확인 + dry-run 계측

연구 질문: **distill된 fact에 memory management(링크·evolution)를 얹으면
naive append보다 나은가?** 대조군은 Nemori 전량 baseline(overall 0.417) —
semantic store만 다르고 나머지는 전부 같다 (단일 변인).

```
Nemori cascade (05와 동일)          A-MEM management (교체된 부분)
발화 → episode → distill  ──────▶  insight가 note로 승격 (K/G/X 생성)
                                    이웃 evo_k=5와 링크·evolution
답변: 질문 embedding → search  ◀──  직접 히트 + 링크 이웃 (m=20 예산)
```

이 노트북은 스모크를 겸하는 dry-run이다 — 실물 확인과 함께 전량 실행
전에 판정해야 할 계측을 남긴다:

- note 실물 — statement가 K/G/X를 얻는 것, 표시 표면은 content뿐인 것
- 링크·evolution — 양방향 링크, 이웃 갱신 빈도 (검증 리뷰 A1·A7)
- **evoke 게이트** — statement 공간에서 baseline과 같은 대역으로 여닫히는지
  (검증 리뷰 A6 실측 → A27 처방; 아래 §3 참고)
- **search 슬롯 점유율** — m=20 예산에서 직접 히트 vs 링크 이웃 (검증 리뷰 A5)

구현: `src/memlab/methods/amem/` + `nemori_amem/` (해석·결정은 각 파일
docstring과 검증 리뷰 A1~A26 — vault의 nemori-amem-review-ledger 참고).

## 준비 — 소켓에 다른 store를 꽂는다

합성의 전부는 조립이다: `NemoriMethod(semantic_store=...)`에 naive
`SemanticStore` 대신 `AmemNoteStore`를 넣는다 (러너·전량 런은
`build_nemori_amem()`이 같은 조립을 한다). 가이드에서는 계측을 위해
store를 `RecordingStore`로 감싼다 — evoke sim과 search 슬롯 구성을
기록만 하고 동작은 그대로 위임한다.

`store._ops`·`store._vectors` 등 내부 상태를 들여다보는 것은 05와 같은
가이드 전용 관례다.

In [1]:
import numpy as np

from memlab.data import load_locomo
from memlab.embedding import cosine_top_k
from memlab.evaluation import set_f1
from memlab.llm import default_provider
from memlab.methods import Utterance
from memlab.methods.amem import AmemNoteStore
from memlab.methods.nemori import NemoriMethod
from memlab.methods.nemori_amem import NemoriAmemConfig

def cos(q, v):
    return float(q @ v / (np.linalg.norm(q) * np.linalg.norm(v)))

class RecordingStore:
    """AmemNoteStore를 감싸 evoke sim·search 슬롯 구성을 기록 (가이드 전용)."""
    def __init__(self, inner):
        self.inner = inner
        self.evoke_log, self.search_log = [], []
    def consolidate(self, insights, occurred_at):
        self.inner.consolidate(insights, occurred_at)
    def evoke(self, query, ks, tau):
        q = np.asarray(query)
        sims = sorted((cos(q, np.asarray(v))
                       for v in self.inner._statement_vectors), reverse=True)[:ks]
        self.evoke_log.append(sims)
        return self.inner.evoke(query, ks, tau)
    def search(self, query, m):
        got = self.inner.search(query, m)
        direct = {self.inner._notes[i].uuid
                  for i in cosine_top_k(self.inner._vectors, np.asarray(query), m)}
        self.search_log.append((sum(n.uuid in direct for n in got), len(got)))
        return got

config = NemoriAmemConfig()
llm = default_provider()
store = AmemNoteStore(llm, evo_k=config.evo_k)  # embed는 양쪽 기본값 동일 (A2는 builder가 보장)
recorder = RecordingStore(store)
method = NemoriMethod(llm, config=config.nemori, semantic_store=recorder)

evolve_log = []  # (링크 건수, 이웃 갱신 건수) — A1·A7 빈도 계측
_orig_evolve = store._ops.evolve
def _logged_evolve(note, neighbors):
    out = _orig_evolve(note, neighbors)
    evolve_log.append((len(out.link_uuids), len(out.neighbor_updates)))
    return out
store._ops.evolve = _logged_evolve

sample = load_locomo()[0]
sessions = {s.index: s for s in sample.sessions}

def ingest_session(session):
    for t in session.turns:
        method.ingest(Utterance(t.speaker, t.text, session.date_time, t.blip_caption))
    print(f"세션 {session.index} ({session.date_time}) — 발화 {len(session.turns)}개"
          f" / 누적 LLM {llm.calls}회 / note {len(store.items)}개")

print(f"{sample.sample_id}: {sample.speaker_a} ↔ {sample.speaker_b}, 세션 {len(sample.sessions)}개")

conv-26: Caroline ↔ Melanie, 세션 19개


## 1. Construct — insight가 note로 승격된다 (A-MEM §3.1)

Nemori cascade는 05와 똑같이 돈다(partition → narrative → distill).
달라지는 것은 distill 산출물의 행선지다: statement가 naive append 대신
CONSTRUCT 콜을 거쳐 keywords(K)·tags(G)·context(X)를 얻는다. 색인은 두
공간이다 — search·이웃 검색은 concat(c,K,G,X) 임베딩(Eq.3), evoke는
statement 임베딩(A27, §3 참고). K/G/X는 임베딩·링크·evolution으로만
일하고 답변 컨텍스트에는 content(=statement 원문)만 나간다 — 통제 실험
결정 (i).

In [2]:
ingest_session(sessions[1])
ingest_session(sessions[2])

for n in store.items[:4]:
    print(f"content : {n.content}")
    print(f"  K {n.keywords}")
    print(f"  G {n.tags}")
    print(f"  X {n.context}\n")

세션 1 (1:56 pm on 8 May, 2023) — 발화 18개 / 누적 LLM 0회 / note 0개


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

세션 2 (1:14 pm on 25 May, 2023) — 발화 17개 / 누적 LLM 14회 / note 4개
content : Caroline attended a LGBTQ support group on May 7, 2023.
  K ['LGBTQ', 'support group', 'attended']
  G ['Support System']
  X Caroline attended a LGBTQ support group on May 7, 2023.

content : Caroline plans to pursue a career in counseling or mental health work.
  K ['counseling', 'mental health', 'career']
  G ['Professional Aspirations', 'Career Goals']
  X Caroline plans to pursue a career in counseling or mental health work.

content : Melanie is the name of Caroline's friend/contact who provided career encouragement.
  K ['Melanie', 'Caroline', 'career encouragement']
  G ['Professional Aspirations', 'Career Goals', 'Support System', 'Personal Relationships', 'Community Involvement']
  X Melanie is the name of Caroline's friend/contact who provided career encouragement. Melanie participated in an organized charity race for mental health support, demonstrating a shared commitment to wellness and community ser

출력에서 볼 것:

- **K/G/X의 결**: keywords는 구체어, tags는 분류어, context는 한 문장
  요약이어야 한다 — 셋이 비슷한 낱말 반복이면 concat 임베딩에 boilerplate가
  실리는 신호다 (§3의 τ 계측과 이어진다)
- **content 무손실** — statement 원문이 그대로 있다. 표시 표면이 content뿐인
  것이 baseline과의 파리티 조건이다
- `[degrade]` 라인이 보이면 construct 실패 → ([], "General", []) 강등이다
  (원본 except 경로 자구) — 발생률이 곧 실측이다

## 2. Evolution — 링크가 생기고 이웃이 갱신된다 (A-MEM §3.2-3.3)

note가 저장될 때마다 이웃 evo_k=5를 회수해 EVOLVE 한 콜로 판정한다:
strengthen(링크 + 새 note 태그 갱신)과 update_neighbor(이웃의 context·
tags 재작성 + 재임베딩). 링크는 양방향이다 — 원본의 죽은 링크 버그를
고친 지점 (store.py docstring).

세션 3~4를 마저 넣고 `end_ingest()`로 잔여 버퍼를 flush한 뒤 실물을 본다.

In [3]:
ingest_session(sessions[3])
ingest_session(sessions[4])
method.end_ingest()

n_links = sum(l for l, _ in evolve_log)
n_updates = sum(u for _, u in evolve_log)
print(f"evolve {len(evolve_log)}콜 — 링크 {n_links}건 / 이웃 갱신 {n_updates}건\n")

linked = next((n for n in store.items if n.links), None)
if linked is None:
    print("링크 0건 — 이번 실행에선 strengthen 판정이 없었다")
else:
    other = next(n for n in store.items if n.uuid == linked.links[0])
    print(f"링크 실물:  {linked.content}")
    print(f"       ↔  {other.content}")
    print(f"  (역방향에도 있나: {linked.uuid in other.links})")

세션 3 (7:55 pm on 9 June, 2023) — 발화 23개 / 누적 LLM 50회 / note 15개


세션 4 (10:37 am on 27 June, 2023) — 발화 18개 / 누적 LLM 66회 / note 19개


evolve 24콜 — 링크 31건 / 이웃 갱신 53건

링크 실물:  Caroline attended a LGBTQ support group on May 7, 2023.
       ↔  Caroline plans to pursue a career in counseling or mental health work.
  (역방향에도 있나: True)


출력에서 볼 것:

- **링크 쌍의 관련성** — 링크된 두 statement가 정말 의미적으로 이어지는지.
  temp 0.7 판정이라 확률적이다 — 0건이어도 이 세션 구간에선 정상 범위
- **이웃 갱신 빈도** (A1·A7 계측) — evolve 콜 대비 갱신 건수. 갱신이 매 콜
  일어나면 paraphrase 드리프트 위험이 실재한다는 뜻이고, 드물면 무변경
  가드가 일하고 있는 것
- **역방향 True** — 양방향 링크가 실제로 걸렸는지

## 3. Evoke 게이트 — statement 공간 (검증 리뷰 A6 → A27)

τ=0.55는 **statement 공간**에서 실측한 경계다 (관련 0.545 이상, 무관
0.505 이하). 초기 설계는 evoke를 concat(c,K,G,X) 벡터로 돌렸는데, K/G/X가
섞이며 분포가 통째로 내려앉아 게이트가 3/14로 닫혔고(A6 실측) cascade가
direct_distill로 붕괴해 수율이 1.07/이벤트로 떨어졌다 (baseline 3.1).

처방(A27): note에 statement 임베딩을 따로 실어 **evoke만 그 공간에서**
돈다. content는 불변이라 evolution의 재임베딩과도 무관하다. 이 셀은
게이트가 baseline(통과 6/7, 통과 sim 0.57~0.79)과 같은 대역에서
여닫히는지 확인한다.

In [4]:
tau = config.nemori.tau
nonempty = [s for s in recorder.evoke_log if s]
n_open = sum(1 for s in nonempty if s[0] > tau)
print(f"evoke {len(recorder.evoke_log)}콜 (store 비어있던 콜 {len(recorder.evoke_log) - len(nonempty)}개)")
print(f"τ={tau} 통과 콜: {n_open}/{len(nonempty)} — 나머지는 direct_distill로 강등\n")

print("콜별 top sim (τ 경계 표시):")
for i, sims in enumerate(nonempty):
    marks = "  ".join(f"{s:.3f}{'✓' if s > tau else '·'}" for s in sims[:5])
    print(f"  evoke#{i}: {marks}")

evoke 8콜 (store 비어있던 콜 1개)
τ=0.55 통과 콜: 6/7 — 나머지는 direct_distill로 강등

콜별 top sim (τ 경계 표시):
  evoke#0: 0.571✓  0.552✓  0.526·
  evoke#1: 0.569✓  0.522·  0.478·  0.465·
  evoke#2: 0.623✓  0.567✓  0.482·  0.474·  0.341·
  evoke#3: 0.674✓  0.578✓  0.547·  0.540·  0.479·
  evoke#4: 0.717✓  0.549·  0.446·  0.445·  0.440·
  evoke#5: 0.535·  0.535·  0.533·  0.530·  0.510·
  evoke#6: 0.576✓  0.568✓  0.555✓  0.549·  0.535·


출력에서 볼 것:

- **통과와 차단의 대역** — 통과 sim은 0.57 이상, 차단은 0.5 안팎 아래로
  갈려야 한다 (A27 프로브 실측: 통과 0.574~0.786, 차단 0.427~0.513).
  통과율이 baseline(6/7)과 같은 수준이면 게이트 파리티 회복이 확인된 것
- 차단된 콜의 상위 항목이 정말 이 episode와 무관한지 눈으로 검증 —
  05 §4의 evoke 경계 셀과 같은 성격의 실측이다

## 4. QA — search 슬롯 점유율 (검증 리뷰 A5)

답변 경로는 Nemori 그대로다(질문 embedding, episodic k=10 + semantic m=20,
상위 r=2 원문 첨부). 다른 것은 semantic m=20이 이 store의 search에서
나온다는 것뿐 — 직접 히트 순위 사이에 히트당 링크 1개를 교차 배치한다.
직접/링크 슬롯 비가 이 결정의 실측이다.

In [5]:
covered = {t.dia_id for i in (1, 2, 3, 4) for t in sessions[i].turns}
exam = [q for q in sample.qa if q.answer and q.evidence and set(q.evidence) <= covered][:4]

for qa in exam:
    pred = method.answer(qa.question)
    direct, total = recorder.search_log[-1]
    print(f"Q: {qa.question}")
    print(f"   gold: {qa.answer}  |  예측: {pred.strip()}  |  set_f1 {set_f1(pred, qa.answer):.2f}")
    print(f"   semantic 슬롯: 직접 {direct} + 링크 {total - direct} = {total}\n")

print(f"총 비용: LLM {llm.calls}회 / {llm.total_tokens:,} 토큰")

Q: When did Caroline go to the LGBTQ support group?
   gold: 7 May 2023  |  예측: May 7, 2023 and June 23, 2023  |  set_f1 0.67
   semantic 슬롯: 직접 20 + 링크 0 = 20



Q: When did Melanie paint a sunrise?
   gold: 2022  |  예측: Last year (2022)  |  set_f1 0.50
   semantic 슬롯: 직접 19 + 링크 1 = 20



Q: What fields would Caroline be likely to pursue in her educaton?
   gold: Psychology, counseling certification  |  예측: Counseling and mental health work.  |  set_f1 0.25
   semantic 슬롯: 직접 20 + 링크 0 = 20



Q: What did Caroline research?
   gold: Adoption agencies  |  예측: Adoption agencies for LGBTQ+ folks.  |  set_f1 0.57
   semantic 슬롯: 직접 20 + 링크 0 = 20

총 비용: LLM 86회 / 88,108 토큰


출력에서 볼 것:

- **링크 슬롯이 몇 개인가** — 0이면 이 구간에선 링크 채널이 답변에 기여할
  기회가 없었던 것 (링크 수 자체가 적으면 자연스럽다). 직접 히트가 절반
  아래로 내려가는 일은 설계상 없어야 한다 (히트당 1개 상한)
- **답변 품질이 05와 같은 결인가** — 같은 질문·같은 답변 경로이므로 큰
  차이가 나면 semantic 회수 내용이 달라졌다는 신호다

## 5. 계측 요약

전량 실행 전 판정에 쓰는 숫자들을 모은다. 이웃 max sim 분포(A24)는
evolve 게이트 변형 실험이 의미 있을지의 재료다 — 이웃이 죄다 낮은 sim이면
게이트가 콜을 크게 아낄 수 있다는 뜻.

In [6]:
notes = store.items
degrees = [len(n.links) for n in notes]
vecs = [np.asarray(n.embedding) for n in notes]
maxsims = [max(cos(v, w) for j, w in enumerate(vecs) if j != i)
           for i, v in enumerate(vecs)] if len(vecs) > 1 else []

print(f"note {len(notes)}개 / 링크 {sum(degrees)}엔트리 (최대 degree {max(degrees, default=0)})")
print(f"evolve {len(evolve_log)}콜 — 링크 {sum(l for l, _ in evolve_log)}건 / "
      f"이웃 갱신 {sum(u for _, u in evolve_log)}건")
ne = [s for s in recorder.evoke_log if s]
print(f"evoke 통과율: {sum(1 for s in ne if s[0] > config.nemori.tau)}/{len(ne)}")
if maxsims:
    print(f"이웃 max sim: median {np.median(maxsims):.3f} / max {max(maxsims):.3f} (A24 재료)")

note 25개 / 링크 62엔트리 (최대 degree 7)
evolve 24콜 — 링크 31건 / 이웃 갱신 53건
evoke 통과율: 6/7
이웃 max sim: median 0.764 / max 0.956 (A24 재료)


## 정리

conv-26 세션 4개로 합성 경로를 실물로 확인했다:

- Nemori cascade는 그대로 돌고, distill 산출물만 note로 승격됐다 (§1)
- 링크·evolution이 실제로 일어나는 것과 그 빈도를 쟀다 (§2)
- evoke 게이트가 statement 공간에서 baseline과 같은 대역으로 여닫히는
  것(§3, A27)과 search 슬롯 구성(§4)을 실측했다

다음: τ 판정 → `uv run memlab-run --method nemori-amem` 전량. 별도
스모크는 없다 — 전량 첫 대화(conv-26 checkpoint)가 스모크 판정을 겸한다
(zep 관례). 대조군은 Nemori 전량 0.417. 판정 기록은 vault의
nemori-amem-review-ledger·runs/ 노트 참고.